##Install and imports

### install

In [ ]:
!pip install bitsandbytes==0.45.5
!pip install transformers==4.40.2
!pip install peft==0.11.1
!pip install accelerate==0.30.1

In [ ]:
!pip install pytrec_eval

### import

In [ ]:
import os
import json
import tqdm
import sys

## Load model

In [ ]:
from transformers import AutoTokenizer, LlamaForCausalLM, AutoModelForCausalLM

model_name = "HuggingFaceH4/zephyr-7b-beta"
tokenizer = AutoTokenizer.from_pretrained(model_name, truncation=True, padding=True, padding_side="left", maximum_length = 2048, model_max_length = 2048)
model = AutoModelForCausalLM.from_pretrained(model_name, load_in_4bit = True, device_map = 'auto')
tokenizer.pad_token = tokenizer.eos_token
model.generation_config.pad_token_id = model.generation_config.eos_token_id

## Query expansion

In [ ]:
import gzip, csv
from datasets import load_dataset
from tqdm.auto import tqdm
import pytrec_eval
from sentence_transformers import CrossEncoder



In [ ]:
# 1. Read qrels
qrels = {}
with open('/content/2019qrels-pass.txt') as f:
    for line in f:
        qid, _, docid, rel = line.strip().split()
        qrels.setdefault(qid, {})[docid] = int(rel)
query_ids = set(qrels.keys())

# 2. Read queries (only keep the 43 in qrels)
queries = {}
with gzip.open('/content/msmarco-test2019-queries.tsv.gz', 'rt', encoding='utf8') as f:
    reader = csv.reader(f, delimiter='\t')
    for qid, text in reader:
        if qid in query_ids:
            queries[qid] = text

In [ ]:
candidates = {}
with gzip.open('/content/msmarco-passagetest2019-top1000.tsv.gz', 'rt', encoding='utf8') as f:
    reader = csv.reader(f, delimiter='\t')
    for row in reader:
        qid, docid = row[0], row[1]
        if qid in query_ids:
            candidates.setdefault(qid, []).append(docid)


In [ ]:
from datasets import load_dataset

# load the passage mapping
passage_ds = load_dataset(
    "sentence-transformers/msmarco-corpus",
    "passage",
    split="train"
)

# we build our dict
passage_text = {
    str(pid): text
    for pid, text in zip(passage_ds["pid"], passage_ds["text"])
}


README.md:   0%|          | 0.00/3.80k [00:00<?, ?B/s]

passage/train-00000-of-00007.parquet:   0%|          | 0.00/238M [00:00<?, ?B/s]

passage/train-00001-of-00007.parquet:   0%|          | 0.00/240M [00:00<?, ?B/s]

passage/train-00002-of-00007.parquet:   0%|          | 0.00/242M [00:00<?, ?B/s]

passage/train-00003-of-00007.parquet:   0%|          | 0.00/243M [00:00<?, ?B/s]

passage/train-00004-of-00007.parquet:   0%|          | 0.00/245M [00:00<?, ?B/s]

passage/train-00005-of-00007.parquet:   0%|          | 0.00/241M [00:00<?, ?B/s]

passage/train-00006-of-00007.parquet:   0%|          | 0.00/240M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8841823 [00:00<?, ? examples/s]

In [ ]:

device = next(model.parameters()).device
expansions = {}
for qid in tqdm(query_ids, desc="Expanding queries"):
    prompt = (
        "Answer the following query:\n\n"
        f"{queries[qid]}\n\n"
        "Give the rationale before answering."
    )
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, padding=True
    ).to(device)
    out = model.generate(**inputs, max_new_tokens=64)
    gen = tokenizer.decode(out[0], skip_special_tokens=True)
    expansions[qid] = gen

# we load our BERT‐style cross‐encoder for reranking
cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2", device=str(device)
)

Expanding queries:   0%|          | 0/43 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/bitsandbytes/nn/modules.py:451: UserWarning: Input type into Linear4bit is torch.float16, but bnb_4bit_compute_dtype=torch.float32 (default). This will lead to slow inference or training speed.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

In [ ]:
cross_encoder # here is our cross encoder which is a BertEncoder

CrossEncoder(
  (model): BertForSequenceClassification(
    (bert): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(30522, 384, padding_idx=0)
        (position_embeddings): Embedding(512, 384)
        (token_type_embeddings): Embedding(2, 384)
        (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-5): 6 x BertLayer(
            (attention): BertAttention(
              (self): BertSelfAttention(
                (query): Linear(in_features=384, out_features=384, bias=True)
                (key): Linear(in_features=384, out_features=384, bias=True)
                (value): Linear(in_features=384, out_features=384, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): BertSelfOutput(
                (dense): Linear(in_features=384, out_features=384, bia

In [ ]:

run_orig = {}
run_exp = {}
for qid in tqdm(all_qids, desc='Scoring runs'):
    docs = candidates.get(qid, [])
    if not docs:
        continue
    # original
    pairs_o = [(queries[qid], passage_text[d]) for d in docs]
    scores_o = cross_encoder.predict(pairs_o, batch_size=64).tolist()
    run_orig[qid] = {d: float(s) for d, s in zip(docs, scores_o)}


    # expanded
    expanded_q = queries[qid] + ' ' + expansions[qid]
    pairs_e = [(expanded_q, passage_text[d]) for d in docs]
    scores_e = cross_encoder.predict(pairs_e, batch_size=64).tolist()
    run_exp[qid] = {d: float(s) for d, s in zip(docs, scores_e)}

common_qids = set(run_orig) & set(run_exp) & all_qids
run_orig_f = {qid: run_orig[qid] for qid in common_qids}
run_exp_f = {qid: run_exp[qid] for qid in common_qids}

if not common_qids:
    raise ValueError(f"No common qids to evaluate: {common_qids}")
evaluator = pytrec_eval.RelevanceEvaluator(
    {qid: qrels[qid] for qid in common_qids},
    {'ndcg_cut.10', 'recall.100', 'map_cut.1000'}
)
metrics_orig = evaluator.evaluate(run_orig_f)
metrics_exp = evaluator.evaluate(run_exp_f)

from statistics import mean

def summarize(metrics):
    return {
        'NDCG@10': mean(m['ndcg_cut_10'] for m in metrics.values()),
        'Recall@100': mean(m['recall_100'] for m in metrics.values()),
        'MAP@1000': mean(m['map_cut_1000'] for m in metrics.values()),
    }

print('Before expansion:', summarize(metrics_orig))
print('After  expansion:', summarize(metrics_exp))


Scoring runs:   0%|          | 0/43 [00:00<?, ?it/s]

Before expansion: {'NDCG@10': 0.7355452132525562, 'Recall@100': 0.5327709290053331, 'MAP@1000': 0.499358660382643}
After  expansion: {'NDCG@10': 0.6696358622346588, 'Recall@100': 0.48597889469057115, 'MAP@1000': 0.4277966445811063}
